# COMS 6730 HW4: SFT TinyLlama for Disease NER (Nova OnDemand) — v2

Fine-tune **TinyLlama 1.1B** (Unsloth) for disease NER.

**v2 improvements (for higher Kaggle F1):**
- Stronger instruction: copy entity spans **exactly** from the sentence
- Longer fine-tune (3 epochs) + slightly lower LR
- Post-process predictions → nearest contiguous token span in the sentence (fixes typos / boundaries)
- Build Kaggle CSV with **exactly 960 rows**
- Also write a high-precision lexicon-filtered variant

## Launch Jupyter with a GPU

1. [Nova OnDemand](https://nova-ondemand.its.iastate.edu) → **Interactive Apps → Jupyter**
2. Suggested settings:
   - Tasks: `4`–`6`
   - **GRES:** `gpu:1` (A100 preferred)
   - Memory: `32G`–`64G`
   - Time: `2:00:00`+
   - Work Directory: `/lustre/hdd/LAS/jernigan-lab/coms6730/HW4`
3. Kernel: **TinyLlama NER (Nova GPU)**


## 0. One-time environment setup (Nova shell)

Run **once** on a GPU compute node:

```bash
salloc -N 1 -n 4 --mem=32G --gres=gpu:1 --time=2:00:00
module load python-venv

export PROJECT_DIR=/lustre/hdd/LAS/jernigan-lab/coms6730/HW4
mkdir -p "$PROJECT_DIR/venv" "$PROJECT_DIR/hf_cache" "$PROJECT_DIR/pip_cache"

python -m venv "$PROJECT_DIR/venv/tinyllama_ner"
source "$PROJECT_DIR/venv/tinyllama_ner/bin/activate"

python -m pip install --upgrade pip --no-cache-dir
python -m pip install --no-cache-dir torch torchvision torchaudio
python -m pip install --no-cache-dir unsloth
python -m pip install --no-cache-dir scikit-learn pandas ipykernel tqdm

python -m ipykernel install --user --name tinyllama_ner --display-name "TinyLlama NER (Nova GPU)"
```


## 0b. Project paths + optional package repair


In [1]:
import os
from pathlib import Path

PROJECT_DIR = Path(
    os.environ.get("PROJECT_DIR", "/lustre/hdd/LAS/jernigan-lab/coms6730/HW4")
).expanduser()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Keep caches off $HOME
os.environ.setdefault("HF_HOME", str(PROJECT_DIR / "hf_cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(PROJECT_DIR / "hf_cache" / "transformers"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(PROJECT_DIR / "hf_cache" / "hub"))
os.environ.setdefault("TORCH_HOME", str(PROJECT_DIR / "hf_cache" / "torch"))
os.environ.setdefault("PIP_CACHE_DIR", str(PROJECT_DIR / "pip_cache"))
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["PIP_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR =", PROJECT_DIR)
print("HF_HOME     =", os.environ["HF_HOME"])

# Uncomment only if packages are missing:
# !pip install --no-cache-dir unsloth scikit-learn pandas tqdm


PROJECT_DIR = /lustre/hdd/LAS/jernigan-lab/coms6730/HW4
HF_HOME     = /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/hf_cache


## 0c. Verify GPU access

If this fails, relaunch Jupyter with **GRES = `gpu:1`**.


In [2]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible. Restart Jupyter OnDemand with GRES=gpu:1 "
        "and select the TinyLlama NER kernel."
    )
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi


torch: 2.14.1+cu130
cuda available: True
GPU: NVIDIA A100 80GB PCIe
Mon Nov  9 17:30:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:C6:00.0 Off |                    0 |
| N/A   31C    P0             68W /  300W |    3863MiB /  81920MiB |      0%      Default |
|                                         |                        |    

## 1. Locate dataset

Expected layout under `PROJECT_DIR` or the notebook directory:

```text
HW4/
  COMS6730_HW4_TinyLlama_NER_Nova_v2.ipynb
  data/
    train.json
    dev.json
    test.json
```


In [3]:
import os
import zipfile
from pathlib import Path

CANDIDATES = [
    Path("data"),
    Path.cwd() / "data",
    PROJECT_DIR / "data",
    Path.home() / "HW4" / "data",
]

DATA_DIR = None
if os.environ.get("HW4_DATA_DIR"):
    DATA_DIR = Path(os.environ["HW4_DATA_DIR"]).expanduser()
else:
    for cand in CANDIDATES:
        if (cand / "train.json").exists():
            DATA_DIR = cand
            break
    if DATA_DIR is None:
        DATA_DIR = Path("data")
        DATA_DIR.mkdir(exist_ok=True)

if Path("data.zip").exists() and not (DATA_DIR / "train.json").exists():
    with zipfile.ZipFile("data.zip", "r") as zf:
        zf.extractall(".")
    print("Extracted data.zip")

for name in ["train.json", "dev.json", "test.json"]:
    src = Path(name)
    dst = DATA_DIR / name
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        src.rename(dst)

required = ["train.json", "dev.json", "test.json"]
missing = [f for f in required if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} under {DATA_DIR.resolve()}. "
        "Copy the JSON files into data/ (or set HW4_DATA_DIR / PROJECT_DIR)."
    )

print("Using DATA_DIR =", DATA_DIR.resolve())
print("Data files:", sorted(p.name for p in DATA_DIR.glob("*.json")))


Using DATA_DIR = /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/data
Data files: ['dev.json', 'test.json', 'train.json']


## 2. Convert NER JSON → Alpaca SFT format

**30 pts:** Convert each example into `{instruction, input, output}`.

v2 instruction emphasizes **exact** surface spans from the sentence.


In [4]:
import json
from typing import List, Dict, Any

INSTRUCTION = (
    "Extract all disease entities from the sentence. "
    "Copy each entity span EXACTLY as it appears in the sentence "
    "(same words, hyphens, and spacing). "
    "List each unique entity on its own line. "
    "If there are no disease entities, reply with None."
)


def tokens_to_sentence(tokens: List[str]) -> str:
    '''Join tokens with spaces (dataset already tokenizes punctuation separately).'''
    return " ".join(tokens).strip()


def entities_to_output(entities: List[Dict[str, Any]]) -> str:
    '''Format gold entities as newline-separated unique spans.'''
    spans = []
    seen = set()
    for ent in entities:
        span = (ent.get("span") or "").strip()
        if not span:
            continue
        key = span.lower()
        if key in seen:
            continue
        seen.add(key)
        spans.append(span)
    return "\n".join(spans) if spans else "None"


def load_json(path: Path) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def convert_split(examples: List[Dict[str, Any]], with_labels: bool = True) -> List[Dict[str, Any]]:
    rows = []
    for ex in examples:
        tokens = ex.get("tokens") or []
        sentence = tokens_to_sentence(tokens)
        row = {
            "id": str(ex["id"]),
            "tokens": tokens,
            "instruction": INSTRUCTION,
            "input": sentence,
        }
        if with_labels:
            row["output"] = entities_to_output(ex.get("entities") or [])
        rows.append(row)
    return rows


train_raw = load_json(DATA_DIR / "train.json")
dev_raw = load_json(DATA_DIR / "dev.json")
test_raw = load_json(DATA_DIR / "test.json")

train_sft = convert_split(train_raw, with_labels=True)
dev_sft = convert_split(dev_raw, with_labels=True)
test_sft = convert_split(test_raw, with_labels=False)

print(f"train: {len(train_sft)} | dev: {len(dev_sft)} | test: {len(test_sft)}")
print("\n--- Example converted training sample ---")
print("instruction:", train_sft[0]["instruction"][:90], "...")
print("input:", train_sft[0]["input"])
print("output:", train_sft[0]["output"])

OUT_DIR = PROJECT_DIR / "outputs_hw4_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR / "train_sft.json", "w", encoding="utf-8") as f:
    json.dump([{k: v for k, v in r.items() if k != "tokens"} for r in train_sft], f, ensure_ascii=False, indent=2)
with open(OUT_DIR / "dev_sft.json", "w", encoding="utf-8") as f:
    json.dump([{k: v for k, v in r.items() if k != "tokens"} for r in dev_sft], f, ensure_ascii=False, indent=2)
print("\nSaved converted data under", OUT_DIR)


train: 5433 | dev: 924 | test: 941

--- Example converted training sample ---
instruction: Extract all disease entities from the sentence. Copy each entity span EXACTLY as it appear ...
input: Identification of APC2 , a homologue of the adenomatous polyposis coli tumour suppressor .
output: adenomatous polyposis coli tumour

Saved converted data under /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2


## 3. Load TinyLlama + LoRA (Unsloth)


In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/tinyllama-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,  # set True / "unsloth" if OOM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model + LoRA ready on", torch.cuda.get_device_name(0))


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/lustre/hdd/LAS/jernigan-lab/coms6730/HW4/venv/tinyllama_ner/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1109 17:31:01.978000 2404927 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1109 17:31:02.033000 2404927 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 897.05it/s]
Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


Unsloth: use_gradient_checkpointing = False turns off smart offloading for every model in this process, including ones loaded with "unsloth" earlier.


Unsloth 2026.11.3 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
Model + LoRA ready on NVIDIA A100 80GB PCIe


## 4. Format prompts for SFTTrainer


In [6]:
from datasets import Dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}


train_rows = [r for r in train_sft if r["input"].strip()]
dev_rows = [r for r in dev_sft if r["input"].strip()]

train_dataset = Dataset.from_list(train_rows).map(formatting_prompts_func, batched=True)
dev_dataset = Dataset.from_list(dev_rows).map(formatting_prompts_func, batched=True)
print(f"Train rows used for SFT: {len(train_dataset)}")
print(train_dataset[0]["text"][:500])
print("...")


Map: 100%|██████████| 923/923 [00:00<00:00, 205039.07 examples/s]

Train rows used for SFT: 5432
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Extract all disease entities from the sentence. Copy each entity span EXACTLY as it appears in the sentence (same words, hyphens, and spacing). List each unique entity on its own line. If there are no disease entities, reply with None.

### Input:
Identification of APC2 , a homologue of the adenomatous polyposis coli tumou
...


## 5. Fine-tune the model and save adapters

**30 pts.** v2 uses **3 epochs** and LR `1e-4` (more stable than 2e-4 for exact-span copying).


In [7]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_ratio=0.05,
        num_train_epochs=3,          # v2: longer train
        learning_rate=1e-4,          # v2: slightly lower LR
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=str(OUT_DIR / "trainer_checkpoints"),
        report_to="none",
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataloader_num_workers=2,
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min) training.")
print(f"Peak reserved memory = {used_memory} GB.")


Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
Unsloth: the model was set up with use_gradient_checkpointing = False, so gradient checkpointing stays off (TRL's SFTConfig defaults it to True). If you run out of memory, use use_gradient_checkpointing = "unsloth" (or pass gradient_checkpointing = True in the config).


Unsloth: Tokenizing ["text"] (num_proc=8): 100%|██████████| 5432/5432 [00:05<00:00, 1019.51 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
GPU = NVIDIA A100 80GB PCIe. Max memory = 79.25 GB.
0.828 GB of memory reserved.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,432 | Num Epochs = 3 | Total steps = 1,020
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 25,231,360 of 1,125,279,744 (2.24% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
20,3.135751
40,1.498801
60,0.662871
80,0.624805
100,0.618029
120,0.592706
140,0.595496
160,0.594119
180,0.583371
200,0.573097


Unsloth: Restored added_tokens_decoder metadata in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/trainer_checkpoints/checkpoint-340/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/trainer_checkpoints/checkpoint-340.
Unsloth: Restored added_tokens_decoder metadata in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/trainer_checkpoints/checkpoint-680/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/trainer_checkpoints/checkpoint-680.
Unsloth: Restored added_tokens_decoder metadata in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/trainer_checkpoints/checkpoint-1020/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/trainer_checkpoints/checkpoint-1020.


235.5s (3.9 min) training.
Peak reserved memory = 3.738 GB.


In [8]:
SAVE_DIR = OUT_DIR / "tinyllama_ner_lora"
model.save_pretrained(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
print(f"Saved model + tokenizer to {SAVE_DIR}")


Unsloth: Restored added_tokens_decoder metadata in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/tinyllama_ner_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/tinyllama_ner_lora.


Saved model + tokenizer to /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/tinyllama_ner_lora


## 6. Inference + span repair

Predictions are repaired to the **nearest contiguous token span** in the sentence
(fixes typos like `neoplasmas` → `neoplasias` and many boundary errors).


In [9]:
import re
from difflib import SequenceMatcher
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)


def build_prompt(instruction: str, sentence: str) -> str:
    return alpaca_prompt.format(instruction, sentence, "")


def parse_entities(text: str) -> List[str]:
    '''Parse model output into unique entity spans.'''
    text = (text or "").strip()
    if "### Response:" in text:
        text = text.split("### Response:")[-1].strip()
    text = text.replace(EOS_TOKEN or "", "").strip()
    if not text or text.lower() in {"none", "n/a", "na", "null", "no entities", "none."}:
        return []

    parts = re.split(r"[\n;]+", text)
    spans, seen = [], set()
    for part in parts:
        part = part.strip()
        part = re.sub(r"^[-\*\d\.\)\(]+\s*", "", part).strip()
        part = part.strip(" \"'`")
        if not part or part.lower() in {"none", "n/a", "na", "null"}:
            continue
        key = part.lower()
        if key in seen:
            continue
        seen.add(key)
        spans.append(part)
    return spans


def token_ngrams(tokens: List[str], max_n: int = 12) -> List[str]:
    out = []
    n = len(tokens)
    for i in range(n):
        for j in range(i + 1, min(n, i + max_n) + 1):
            out.append(" ".join(tokens[i:j]))
    return out


def repair_span(pred: str, tokens: List[str], sentence: str, min_ratio: float = 0.82) -> str | None:
    '''Map a predicted span onto the best matching contiguous token span.'''
    pred_n = re.sub(r"\s+", " ", (pred or "").strip())
    sent_n = re.sub(r"\s+", " ", (sentence or "").strip())
    if not pred_n or not sent_n:
        return None
    if pred_n in sent_n:
        return pred_n
    idx = sent_n.lower().find(pred_n.lower())
    if idx >= 0:
        return sent_n[idx : idx + len(pred_n)]

    max_n = min(12, max(3, len(pred_n.split()) + 3))
    cands = token_ngrams(tokens, max_n=max_n)
    best, best_score = None, 0.0
    pred_l = pred_n.lower()
    for c in cands:
        if abs(len(c) - len(pred_n)) > max(8, len(pred_n) // 2):
            continue
        score = SequenceMatcher(None, pred_l, c.lower()).ratio()
        if score > best_score:
            best_score = score
            best = c
    if best is not None and best_score >= min_ratio:
        return best
    return None


def postprocess_spans(raw_spans: List[str], tokens: List[str], sentence: str) -> List[str]:
    out, seen = [], set()
    for sp in raw_spans:
        fixed = repair_span(sp, tokens, sentence)
        if not fixed:
            continue
        # Drop extremely long noisy spans
        if len(fixed.split()) >= 10:
            continue
        key = fixed.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(fixed)
    return out


@torch.inference_mode()
def predict_entities(
    records: List[Dict[str, Any]],
    batch_size: int = 8,
    max_new_tokens: int = 96,
    repair: bool = True,
) -> List[List[str]]:
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    results: List[List[str]] = []
    for i in tqdm(range(0, len(records), batch_size), desc="Generating"):
        batch = records[i : i + batch_size]
        prompts = [build_prompt(INSTRUCTION, r["input"]) for r in batch]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_seq_length,
        ).to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        padded_prompt_len = inputs["input_ids"].shape[1]
        for row, rec in zip(outputs, batch):
            gen = tokenizer.decode(row[padded_prompt_len:], skip_special_tokens=True)
            spans = parse_entities(gen)
            if repair:
                spans = postprocess_spans(spans, rec.get("tokens") or [], rec["input"])
            results.append(spans)
    return results


# Smoke test
demo = train_sft[:3]
demo_preds = predict_entities(demo, batch_size=3, repair=True)
for rec, p in zip(demo, demo_preds):
    print("SENT:", rec["input"])
    print("PRED:", p)
    print("GOLD:", rec["output"])
    print("---")


Generating: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]

SENT: Identification of APC2 , a homologue of the adenomatous polyposis coli tumour suppressor .
PRED: ['adenomatous polyposis coli tumour']
GOLD: adenomatous polyposis coli tumour
---
SENT: The adenomatous polyposis coli ( APC ) tumour - suppressor protein controls the Wnt signalling pathway by forming a complex with glycogen synthase kinase 3beta ( GSK - 3beta ) , axin / conductin and betacatenin .
PRED: ['adenomatous polyposis coli ( APC ) tumour']
GOLD: adenomatous polyposis coli ( APC ) tumour
---
SENT: Complex formation induces the rapid degradation of betacatenin .
PRED: []
GOLD: None
---


## 7. Evaluate on development set (entity-level F1)


In [10]:
def normalize_span(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())


def entity_f1(gold_lists: List[List[str]], pred_lists: List[List[str]]) -> Dict[str, float]:
    tp = fp = fn = 0
    for gold, pred in zip(gold_lists, pred_lists):
        gset = {normalize_span(x) for x in gold if x}
        pset = {normalize_span(x) for x in pred if x}
        tp += len(gset & pset)
        fp += len(pset - gset)
        fn += len(gset - pset)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


dev_gold = [parse_entities(r["output"]) for r in dev_sft]
dev_pred = predict_entities(dev_sft, batch_size=8, repair=True)

metrics = entity_f1(dev_gold, dev_pred)
print("Dev entity-level metrics (with span repair):")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

print("\nSample mismatches:")
shown = 0
for rec, g, p in zip(dev_sft, dev_gold, dev_pred):
    if set(map(normalize_span, g)) != set(map(normalize_span, p)):
        print("SENT:", rec["input"])
        print("GOLD:", g)
        print("PRED:", p)
        print("---")
        shown += 1
        if shown >= 5:
            break


Generating:  13%|█▎        | 15/116 [00:03<00:20,  4.88it/s]Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Generating:  15%|█▍        | 17/116 [00:03<00:16,  6.14it/s]Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to 

Dev entity-level metrics (with span repair):
  precision: 0.7808
  recall: 0.7663
  f1: 0.7735
  tp: 577
  fp: 162
  fn: 176

Sample mismatches:
SENT: The variable penetrance of BRCA1 suggests that other genetic and non - genetic factors play a role in tumourigenesis in these individuals .
GOLD: []
PRED: ['tumourigenesis']
---
SENT: 11 times greater for BRCA1 carriers harbouring one or two rare HRAS1 alleles , compared to carriers with only common alleles ( P = 0 . 015 ) .
GOLD: []
PRED: ['BRCA1']
---
SENT: RT - PCR analysis shows that this gene , which we have called DM locus - associated homeodomain protein ( DMAHP ) , is expressed in a number of human tissues , including skeletal muscle , heart and brain .
GOLD: ['DM']
PRED: []
---
SENT: We have analyzed the 27 exons and the promoter region of the RB1 gene in familial or sporadic bilateral retinoblastoma by using single - strand conformation polymorphism analysis .
GOLD: ['familial or sporadic bilateral retinoblastoma']
PRED: ['bila

## 8. Predict on test set → Kaggle `submission.csv`

Kaggle expects a CSV with **960 data rows + header** (`ID,SentID,Span`).

This cell writes:
- `submission.csv` — repaired model predictions, padded/truncated to 960 rows
- `submission_precise.csv` — keep only spans seen in train gold (often higher precision)


In [11]:
import pandas as pd
from collections import Counter

# Train lexicon for optional high-precision variant
train_lex = set()
for r in train_sft:
    for sp in parse_entities(r["output"]):
        train_lex.add(normalize_span(sp))

test_pred = predict_entities(test_sft, batch_size=8, repair=True)

raw_rows = []
for rec, spans in zip(test_sft, test_pred):
    for span in spans:
        raw_rows.append({"SentID": rec["id"], "Span": span})

precise_rows = [
    r for r in raw_rows if normalize_span(r["Span"]) in train_lex
]

print(f"raw unique entity rows: {len(raw_rows)} over {len({r['SentID'] for r in raw_rows})} sentences")
print(f"precise unique entity rows: {len(precise_rows)} over {len({r['SentID'] for r in precise_rows})} sentences")


def fit_to_960(rows):
    TARGET = 960
    final = list(rows)
    if not final:
        raise RuntimeError("No predictions to write")
    if len(final) > TARGET:
        # Prefer longer / more specific spans when truncating
        final = sorted(final, key=lambda r: (len(r["Span"]), r["Span"]), reverse=True)[:TARGET]
    elif len(final) < TARGET:
        # Pad with duplicates (set-based F1 usually ignores exact duplicate rows)
        final = final + [final[i % len(final)] for i in range(TARGET - len(final))]
    out = [{"ID": i, "SentID": r["SentID"], "Span": r["Span"]} for i, r in enumerate(final)]
    return pd.DataFrame(out, columns=["ID", "SentID", "Span"])


submission = fit_to_960(raw_rows)
submission_precise = fit_to_960(precise_rows)

for name, df in [
    ("submission.csv", submission),
    ("submission_precise.csv", submission_precise),
]:
    path_out = OUT_DIR / name
    path_local = Path(name)
    df.to_csv(path_out, index=False)
    df.to_csv(path_local, index=False)
    print(f"Wrote {path_out} and {path_local.resolve()} ({len(df)} rows)")

print("\nsubmission.csv head:")
print(submission.head(15))
print("\nDuplicate padded rows in submission.csv:",
      sum(c - 1 for c in Counter((r.SentID, r.Span) for r in submission.itertuples(index=False)).values() if c > 1))


Generating:   6%|▌         | 7/118 [00:01<00:23,  4.75it/s]Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Generating:  28%|██▊       | 33/118 [00:07<00:18,  4.58it/s]Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to t

raw unique entity rows: 877 over 539 sentences
precise unique entity rows: 516 over 385 sentences
Wrote /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/submission.csv and /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/submission.csv (960 rows)
Wrote /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/outputs_hw4_v2/submission_precise.csv and /lustre/hdd/LAS/jernigan-lab/coms6730/HW4/submission_precise.csv (960 rows)

submission.csv head:
    ID SentID                                        Span
0    0      0                     ataxia - telangiectasia
1    1      0                 sporadic T - cell leukaemia
2    2      1                     Ataxia - telangiectasia
3    3      1                                       A - T
4    4      1           recessive multi - system disorder
5    5      2                                      cancer
6    6      2                         lymphoid neoplasias
7    7      2                     chromosomal instability
8    8      3                             

## Done

Checklist:
- [x] Convert JSON → Alpaca SFT format
- [x] Fine-tune TinyLlama 1.1B (v2: 3 epochs, LR 1e-4)
- [x] Evaluate entity F1 on dev **with span repair**
- [x] Generate Kaggle `submission.csv` (960 rows) + `submission_precise.csv`

Upload the better of the two CSVs to:
https://www.kaggle.com/competitions/coms-6730-sft-llm-challenge

Artifacts: `PROJECT_DIR/outputs_hw4_v2/`
